# GBM snRNA-seq 处理与跨数据集整合

**数据来源:**
- **GSE286419** (Pai et al. 2025): 4名 IDHwt GBM 患者，Tumor + Peritumoral，snRNA（独立实验）
- **GSE276177** (Sojka et al. 2025): 3名 IDHwt GBM 患者，Tumor + Peritumoral，10x Multiome（snRNA 部分）

**分析流程:**
1. 数据加载与 QC
2. 归一化、HVG 选择、PCA
3. 细胞类型注释（marker gene 手动 + 自动)
4. 跨数据集整合（Harmony 批次校正）→ 输出 `results/1_snRNA_processing/adata1.h5ad`

> 多模态整合（snRNA + snATAC）请见 **3_multiomics_integration.ipynb**

**QC 标准:**
- 统一采用相同标准 (nFeature>400, nCount>1000, MT%<15%, nFeature<10000)

## 0. 环境配置

In [ ]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gc
import warnings
warnings.filterwarnings('ignore')
import muon as mu
from muon import atac as ac
import scrublet as scr
import snapatac2 as snap

In [ ]:
print(f"scanpy: {sc.__version__}")
print(f"anndata: {ad.__version__}")
print(f"muon: {mu.__version__}")

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)
sc.settings.fig_dir = './figures/1_snRNA_processing/'
os.makedirs('./figures/1_snRNA_processing/', exist_ok=True)
os.makedirs('./results/1_snRNA_processing/', exist_ok=True)
os.makedirs('./figures/', exist_ok=True)
os.makedirs('./results/', exist_ok=True)

## 1. 数据概览与路径定义

In [ ]:
base_dir = './local_inputs'
RNA_DIR = os.path.join(base_dir, 'GSE286419_snRNA_RAW')
ATAC_DIR = os.path.join(base_dir, 'GSE286415_snATAC_RAW')

# 样本信息（基于 GSE286419_GPL24676_sample_anno.csv）
SAMPLE_INFO = pd.read_csv(os.path.join(base_dir, 'sample_info.csv')).set_index('sample').to_dict('index')

print(f"共 {len(SAMPLE_INFO)} 个样本，4 个患者 (four study-local patients)")
print(f"snRNA 数据目录: {RNA_DIR}")
print(f"snATAC 数据目录: {ATAC_DIR}")

In [ ]:
anno_df = pd.read_csv(os.path.join(RNA_DIR, 'GSE286419_GPL24676_sample_anno.csv'), index_col=0)
display(anno_df)

In [ ]:
GSE276177_DIR = os.path.join(base_dir, 'GSE276177_RAW')

# GSE276177 样本信息（10x Multiome: 同一核同时 RNA+ATAC）
GSE276177_SAMPLES = pd.read_csv(os.path.join(base_dir, 'snrna_sample_info.csv')).set_index('sample').to_dict('index')
anno_276177 = pd.read_csv(os.path.join(GSE276177_DIR, 'GSE276177_GPL24676_sample_anno.csv'), index_col=0)
display(anno_276177)
print(f"\nGSE276177: {len(GSE276177_SAMPLES)} 个样本 (Multiome)")

## 2. GSE286419 snRNA-seq 处理（Pai et al. 2025）

### 2.1 加载数据

In [ ]:
rna_adatas = {}

for sample_name, info in SAMPLE_INFO.items():
    gsm = info['rna_gsm']
    prefix = f"{gsm}_{sample_name}"
    
    mtx_path = os.path.join(RNA_DIR, f"{prefix}_matrix.mtx.gz")
    bc_path = os.path.join(RNA_DIR, f"{prefix}_barcodes.tsv.gz")
    feat_path = os.path.join(RNA_DIR, f"{prefix}_features.tsv.gz")
    
    if not os.path.exists(mtx_path):
        print(f"⚠ {sample_name}: 文件不存在，跳过")
        continue
    
    adata = sc.read_mtx(mtx_path).T
    
    barcodes = pd.read_csv(bc_path, header=None, sep='\t')
    adata.obs_names = [f"{sample_name}_{bc}" for bc in barcodes[0].values]
    
    features = pd.read_csv(feat_path, header=None, sep='\t')
    adata.var_names = features[1].values
    if features.shape[1] > 0:
        adata.var['gene_ids'] = features[0].values
    
    adata.var_names_make_unique()
    
    adata.obs['sample'] = sample_name
    adata.obs['patient'] = sample_name[:4]
    adata.obs['tissue'] = info['tissue']
    adata.obs['progress'] = info['progress']
    adata.obs['age'] = info['age']
    adata.obs['modality'] = 'snRNA'
    
    rna_adatas[sample_name] = adata
    print(f"✓ {sample_name}: {adata.n_obs} cells × {adata.n_vars} genes")

print(f"\n共加载 {len(rna_adatas)} 个 snRNA-seq 样本")

In [ ]:
rna = ad.concat(list(rna_adatas.values()), merge='same')
rna.obs_names_make_unique()
print(f"合并后: {rna.n_obs} cells × {rna.n_vars} genes")
del rna_adatas
gc.collect()

### 2.2 质量控制（QC）

In [ ]:
rna.var['mt'] = rna.var_names.str.startswith('MT-')
rna.var['ribo'] = rna.var_names.str.startswith(('RPS', 'RPL'))

sc.pp.calculate_qc_metrics(rna, qc_vars=['mt', 'ribo'],
                           percent_top=None, log1p=False, inplace=True)

print("QC 指标统计:")
print(rna.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].describe())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, metric in enumerate(['n_genes_by_counts', 'total_counts', 'pct_counts_mt']):
    sc.pl.violin(rna, metric, groupby='sample', rotation=45,
                 ax=axes[0, idx], show=False)
    axes[0, idx].set_title(metric)

sc.pl.scatter(rna, x='total_counts', y='n_genes_by_counts',
              color='pct_counts_mt', ax=axes[1, 0], show=False)
axes[1, 0].axhline(y=400, color='r', linestyle='--', label='min genes=400')
axes[1, 0].axhline(y=10000, color='r', linestyle='--', label='max genes=10000')
axes[1, 0].axvline(x=1000, color='b', linestyle='--', label='min counts=1000')
axes[1, 0].legend(fontsize=8)

sc.pl.scatter(rna, x='total_counts', y='pct_counts_mt', ax=axes[1, 1], show=False)
axes[1, 1].axhline(y=15, color='r', linestyle='--', label='max MT%=15')
axes[1, 1].legend()

sample_counts = rna.obs['sample'].value_counts()
sample_counts.plot.bar(ax=axes[1, 2], color='steelblue')
axes[1, 2].set_title('Cells per sample')
axes[1, 2].set_ylabel('Number of cells')

plt.tight_layout()
plt.savefig('./figures/1_snRNA_processing/snRNA_QC_before_filter.pdf', bbox_inches='tight')
plt.show()

In [ ]:
print(f"过滤前: {rna.n_obs} cells")

# 统一使用显式布尔索引，避免混合 sc.pp.filter_cells 与手动索引
keep = (
    (rna.obs['n_genes_by_counts'] >= 400) &
    (rna.obs['n_genes_by_counts'] < 10000) &
    (rna.obs['total_counts'] > 1000) &
    (rna.obs['pct_counts_mt'] < 15)
)
n_removed = rna.n_obs - keep.sum()
print(f"Cell QC 过滤 (nGene∈[400,10000), nCount>1000, MT%<15%): 去除 {n_removed} cells")
rna = rna[keep, :].copy()

# 基因过滤: 至少在3个cell中检测到
gene_mask = (rna.X > 0).sum(axis=0).A1 >= 3 if hasattr(rna.X, 'A1') else (rna.X > 0).sum(axis=0) >= 3
n_gene_removed = rna.n_vars - gene_mask.sum()
rna = rna[:, gene_mask].copy()
print(f"低表达基因过滤 (min_cells=3): 去除 {n_gene_removed} genes")
print(f"过滤后: {rna.n_obs} cells × {rna.n_vars} genes")


### 2.3 Doublet 检测（Scrublet）

In [ ]:
doublet_scores = []
predicted_doublets = []

for sample in rna.obs['sample'].unique():
    mask = rna.obs['sample'] == sample
    adata_sub = rna[mask].copy()
    
    scrub = scr.Scrublet(adata_sub.X, expected_doublet_rate=0.06)
    doublet_score, predicted = scrub.scrub_doublets(
        min_counts=2, min_cells=3, min_gene_variability_pctl=85,
        n_prin_comps=30, verbose=False
    )
    
    doublet_scores.extend(doublet_score)
    predicted_doublets.extend(predicted)
    n_doublets = sum(predicted)
    print(f"{sample}: {n_doublets}/{adata_sub.n_obs} doublets "
          f"({n_doublets/adata_sub.n_obs*100:.1f}%)")

rna.obs['doublet_score'] = doublet_scores
rna.obs['predicted_doublet'] = predicted_doublets

n_before = rna.n_obs
rna = rna[~rna.obs['predicted_doublet']].copy()
print(f"\n移除 doublet: {n_before} → {rna.n_obs} cells "
      f"(去除 {n_before - rna.n_obs})")

### 2.4 保存原始 counts

标准化和 HVG 选择将在 步骤4 合并后统一进行，此处仅保存 raw counts。

In [ ]:
# 保存原始 counts（标准化将在 步骤4 合并后统一进行）
if 'counts' not in rna.layers:
    rna.layers['counts'] = rna.X.copy()

print(f"原始 counts 已保存: {rna.shape}")
print("标准化/HVG/PCA/Harmony 将在 步骤4 合并后统一进行")

## 3. GSE276177 snRNA-seq 处理（Sojka et al. 2025）

**数据特点:** 10x Multiome（同核同时产出 snRNA + snATAC，共享 barcode）

### 3.1 加载数据（.h5 格式）

In [ ]:
rna_276177_adatas = {}

for sample_name, info in GSE276177_SAMPLES.items():
    h5_path = os.path.join(GSE276177_DIR, info['rna_file'])
    
    if not os.path.exists(h5_path):
        print(f"⚠ {sample_name}: {info['rna_file']} 不存在")
        continue
    
    adata = sc.read_10x_h5(h5_path)
    adata.var_names_make_unique()
    
    # 添加元数据
    adata.obs_names = [f"S276177_{sample_name}_{bc}" for bc in adata.obs_names]
    adata.obs['sample'] = sample_name
    adata.obs['patient'] = info['patient']
    adata.obs['tissue'] = info['tissue']
    adata.obs['sex'] = info['sex']
    adata.obs['genotype'] = info['genotype']
    adata.obs['dataset'] = 'GSE276177'
    adata.obs['modality'] = 'snRNA'
    adata.obs['data_type'] = 'Multiome'  # 标记为 Multiome 数据
    
    rna_276177_adatas[sample_name] = adata
    print(f"✓ {sample_name}: {adata.n_obs} cells × {adata.n_vars} genes")

print(f"\n共加载 {len(rna_276177_adatas)} 个 GSE276177 snRNA 样本")

In [ ]:
# 合并 GSE276177 snRNA 样本
rna_276177 = ad.concat(list(rna_276177_adatas.values()), merge='same')
rna_276177.obs_names_make_unique()
print(f"GSE276177 合并: {rna_276177.n_obs} cells × {rna_276177.n_vars} genes")
del rna_276177_adatas
gc.collect()

### 3.2 质量控制

In [ ]:
# 计算 QC 指标
rna_276177.var['mt'] = rna_276177.var_names.str.startswith('MT-')
rna_276177.var['ribo'] = rna_276177.var_names.str.startswith(('RPS', 'RPL'))

sc.pp.calculate_qc_metrics(rna_276177, qc_vars=['mt', 'ribo'],
                           percent_top=None, log1p=False, inplace=True)

print("GSE276177 QC 统计:")
print(rna_276177.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].describe())

In [ ]:
# QC 可视化
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sc.pl.violin(rna_276177, 'n_genes_by_counts', groupby='sample', rotation=45,
             ax=axes[0], show=False)
axes[0].set_title('nGenes (GSE276177)')

sc.pl.violin(rna_276177, 'total_counts', groupby='sample', rotation=45,
             ax=axes[1], show=False)
axes[1].set_title('nCounts (GSE276177)')

sc.pl.violin(rna_276177, 'pct_counts_mt', groupby='sample', rotation=45,
             ax=axes[2], show=False)
axes[2].set_title('MT% (GSE276177)')

plt.tight_layout()
plt.savefig('./figures/1_snRNA_processing/GSE276177_snRNA_QC.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# GSE276177 过滤（兼容 Sojka & Pai 标准）
# 使用更严格的 Sojka 标准 (TSS>5, fragments>=3500) 但对 RNA 使用通用标准
print(f"过滤前: {rna_276177.n_obs} cells")

# 统一使用显式布尔索引
keep = (
    (rna_276177.obs['n_genes_by_counts'] >= 400) &
    (rna_276177.obs['n_genes_by_counts'] < 10000) &
    (rna_276177.obs['total_counts'] > 1000) &
    (rna_276177.obs['pct_counts_mt'] < 15)
)
n_removed = rna_276177.n_obs - keep.sum()
print(f"Cell QC 过滤 (nGene∈[400,10000), nCount>1000, MT%<15%): 去除 {n_removed} cells")
rna_276177 = rna_276177[keep, :].copy()

# 基因过滤
gene_mask = (rna_276177.X > 0).sum(axis=0).A1 >= 3 if hasattr(rna_276177.X, 'A1') else (rna_276177.X > 0).sum(axis=0) >= 3
n_gene_removed = rna_276177.n_vars - gene_mask.sum()
rna_276177 = rna_276177[:, gene_mask].copy()
print(f"低表达基因过滤 (min_cells=3): 去除 {n_gene_removed} genes")
print(f"最终: {rna_276177.n_obs} cells × {rna_276177.n_vars} genes")


### 3.3 Doublet 检测

In [ ]:
# Scrublet doublet 检测（自动阈值 + 安全上限）
d_scores = []
d_predicted = []

for sample in rna_276177.obs['sample'].unique():
    mask = rna_276177.obs['sample'] == sample
    sub = rna_276177[mask].copy()
    
    scrub = scr.Scrublet(sub.X, expected_doublet_rate=0.06)
    score, pred = scrub.scrub_doublets(
        min_counts=2, min_cells=3, min_gene_variability_pctl=85,
        n_prin_comps=30, verbose=False
    )
    
    # 如果自动阈值导致 >20% doublets，使用手动阈值 0.25
    auto_rate = sum(pred) / sub.n_obs
    if auto_rate > 0.20:
        manual_threshold = 0.25
        pred = score > manual_threshold
        print(f"{sample}: 自动阈值异常 ({auto_rate:.1%})，"
              f"改用手动阈值 {manual_threshold} → {sum(pred)}/{sub.n_obs} ({sum(pred)/sub.n_obs:.1%})")
    else:
        print(f"{sample}: {sum(pred)}/{sub.n_obs} doublets ({auto_rate:.1%})")
    
    d_scores.extend(score)
    d_predicted.extend(pred)

In [ ]:
rna_276177.obs['doublet_score'] = d_scores
rna_276177.obs['predicted_doublet'] = d_predicted

n_before = rna_276177.n_obs
rna_276177 = rna_276177[~rna_276177.obs['predicted_doublet']].copy()
print(f"\n移除 doublet: {n_before} → {rna_276177.n_obs} cells")

## 4. snRNA-seq 跨数据集整合

将两个数据集的 snRNA 合并，进行联合降维与分析。

### 4.1 合并数据集

In [ ]:
# 为 GSE286419 添加统一标签
rna.obs['dataset'] = 'GSE286419'
rna.obs['data_type'] = 'Independent'

# GSE276177 保存 raw counts
if 'counts' not in rna_276177.layers:
    rna_276177.layers['counts'] = rna_276177.X.copy()

In [ ]:
# 两个数据集的共同基因
common_genes = list(
    set(rna.var_names) & set(rna_276177.var_names)
)
print(f'共同基因数: {len(common_genes)}')

rna_sub = rna[:, common_genes].copy()
rna_276177_sub = rna_276177[:, common_genes].copy()

In [ ]:
# 合并两个数据集（使用 raw counts）
rna_merged = ad.concat([rna_sub, rna_276177_sub], merge='same')
rna_merged.obs_names_make_unique()

# 统一标准化
rna_merged.layers['counts'] = rna_merged.X.copy()
sc.pp.normalize_total(rna_merged, target_sum=1e4)
sc.pp.log1p(rna_merged)
rna_merged.raw = rna_merged.copy()

In [ ]:
print(f'合并后: {rna_merged.n_obs} cells × {rna_merged.n_vars} genes')
print(f'  GSE286419: {(rna_merged.obs["dataset"] == "GSE286419").sum()} cells')
print(f'  GSE276177: {(rna_merged.obs["dataset"] == "GSE276177").sum()} cells')
print('统一标准化完成')

del rna_sub, rna_276177_sub
gc.collect()

In [ ]:
# 所有数据集在源头已统一: Tumor / Peritumoral
rna_merged.obs['tissue_unified'] = rna_merged.obs['tissue'].copy()
print('tissue 标签:')
print(rna_merged.obs['tissue_unified'].value_counts())


### 4.2 联合 HVG + PCA + Harmony

In [ ]:
# 重新选择 HVG（跨数据集，n=2000，对齐参考标准）
try:
    sc.pp.highly_variable_genes(rna_merged, n_top_genes=2000, flavor='seurat_v3',
                                layer='counts', batch_key='sample')
    hvg_flavor_used = 'seurat_v3'
except (ValueError, Exception):
    print('⚠ seurat_v3 失败，回退到 seurat_v3_paper')
    sc.pp.highly_variable_genes(rna_merged, n_top_genes=2000, flavor='seurat_v3_paper',
                                layer='counts', batch_key='sample', span=0.5)
    hvg_flavor_used = 'seurat_v3_paper'

print(f"跨数据集 HVG ({hvg_flavor_used}): {rna_merged.var['highly_variable'].sum()}")

In [ ]:
# PCA + Harmony（按 sample 校正，同时处理跨数据集批次效应）
rna_hvg_m = rna_merged[:, rna_merged.var['highly_variable']].copy()

# 回归去除测序深度和 MT 比例影响（对齐参考标准）
sc.pp.regress_out(rna_hvg_m, ['total_counts', 'pct_counts_mt'])

sc.pp.scale(rna_hvg_m, max_value=10)
sc.pp.pca(rna_hvg_m, n_comps=50, random_state=42)

rna_merged.obsm['X_pca'] = rna_hvg_m.obsm['X_pca']
del rna_hvg_m
gc.collect()


# Harmony 校正（同时考虑 sample 和 dataset）print("Harmony 批次校正完成")

sc.external.pp.harmony_integrate(rna_merged, key='sample', basis='X_pca',adjusted_basis='X_pca_harmony')

### 4.3 UMAP + 聚类

In [ ]:
# UMAP + 聚类
sc.pp.neighbors(rna_merged, use_rep='X_pca_harmony', n_pcs=30, n_neighbors=30, random_state=42)
sc.tl.umap(rna_merged, min_dist=0.3, random_state=42)

sc.tl.leiden(rna_merged, resolution=0.5, key_added='leiden_0.5',
             flavor='igraph', n_iterations=2, random_state=42)
print(f"Leiden (res=0.5): {rna_merged.obs['leiden_0.5'].nunique()} clusters")

In [ ]:
# 聚类 UMAP 可视化（注释前）
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
sc.pl.umap(rna_merged, color='leiden_0.5', ax=axes[0], show=False,
           title='Clusters', legend_loc='on data', legend_fontsize=5)
sc.pl.umap(rna_merged, color='dataset', ax=axes[1], show=False,
           title='Dataset')
sc.pl.umap(rna_merged, color='tissue_unified', ax=axes[2], show=False,
           title='Tissue')
plt.tight_layout()
plt.show()

### 4.4 跨数据集细胞类型注释

In [ ]:
# 在合并数据上重新进行 marker-based 注释
marker_gene_sets = {
    # --- Tumor & Glia ---
    # "Tumor": ["GFAP", "PTPRZ1", "SCG3", "C1orf61", "SOX2", "SOX6", "BCAN", "NOVA1", "NRCAM", "TUBB2B"],
    # "Prol.tumor": ["SOX2", "SOX6", "MKI67", "TOP2A", "CCDC34", "PBK", "UBE2T", "CENPV", "CENPF"],
    "Oligodendrocyte": ["MAG", "CLDN11", "APLP1", "TMEM144", "CNDP1", "EDIL3", "LARP6", "MOG", "AMER2", "TUBB4A"],
    "MES-like": ["VIM", "CD44", "CHI3L1", "HILPDA", "DDIT3", "ENO2"],
    "AC-like": ["GFAP", "S100B", "HOPX", "SLC1A3", "MLC1"],
    "OPC-like": ["PLP1", "OLIG1", "OMG", "TNR", "ALCAM"],
    "NPC-like": ["SOX4", "SOX11", "DCX", "CD24", "STMN1", "STMN2"],
    
    # --- Lymphoid ---
    "NK/T": ["CD3D", "CD3E", "TRBC2", "CD3G", "CD2", "TRAC", "IL7R", "TRBC1", "LCK", "SKAP1", "CD48"],
    "Prol.NK/T": ["MKI67", "TOP2A", "CD3D", "CD3E", "TRBC2", "CD3G"],

    # --- Vascular / Stromal ---
    "Pericyte": ["COL1A2", "COL6A2", "PDGFRB", "COL3A1", "EDNRA", "LUM", "COL1A1", "PLAC9", "FRZB"],
    "Endothelial": ["ESAM", "FLT1", "RAMP2", "VWF", "EGFL7", "SLC9A3R2", "CAVIN2", "ADGRL4", "ABLIM1", "ABCB1"],

    # --- Myeloid & Dendritic ---
    # "Dendritic": ["AREG", "FCER1A", "CLEC10A", "HLA-DQB1", "HLA-DQA1"],
    "Monocyte": ["FCN1", "VCAN", "S100A8", "S100A9"],
    "Microglia": ["TMEM119", "CX3CR1", "TREM2", "GPNMB", "CTSD", "CD68"], 
    "Macrophage": ["SPP1", "APOE", "GPNMB", "CTSD", "CD68", "C1QA", "C1QB"],
    "CDC": ["FCER1A", "HLA-DPB1", "CD83", "AREG"],
    "Neutrophil": ["S100A8", "FCGR3B", "IFITM2"],
}

In [ ]:
# 检查基因集重叠并警告
from collections import Counter
all_genes = []
for ct, genes in marker_gene_sets.items():
    all_genes.extend([(g, ct) for g in genes])
gene_cts = {}
for g, ct in all_genes:
    gene_cts.setdefault(g, []).append(ct)
overlapping = {g: cts for g, cts in gene_cts.items() if len(cts) > 1}
if overlapping:
    print("⚠️ 基因集重叠警告:")
    for g, cts in overlapping.items():
        print(f"  {g} → {', '.join(cts)}")

In [ ]:
# 打分
for ct, genes in marker_gene_sets.items():
    present = [g for g in genes if g in rna_merged.var_names]
    if present:
        sc.tl.score_genes(rna_merged, gene_list=present, score_name=f'score_{ct}')

In [ ]:
score_cols = [c for c in rna_merged.obs.columns if c.startswith('score_')]
scores_df = rna_merged.obs[score_cols]

# 置信度阈值: top1 - top2 的分差
MIN_SCORE_DIFF = 0.05  # top1 与 top2 的最小分差
sorted_scores = scores_df.values.copy()
sorted_scores.sort(axis=1)
top1 = sorted_scores[:, -1]
top2 = sorted_scores[:, -2]
score_diff = top1 - top2

In [ ]:
predicted = scores_df.idxmax(axis=1).str.replace('score_', '')
low_confidence = score_diff < MIN_SCORE_DIFF
predicted[low_confidence] = 'Ambiguous'
rna_merged.obs['predicted_celltype'] = predicted
rna_merged.obs['annotation_confidence'] = score_diff

In [ ]:
n_ambig = low_confidence.sum()
print(f"\n注释结果: {(~low_confidence).sum()} 高置信度, {n_ambig} Ambiguous ({n_ambig/len(low_confidence)*100:.1f}%)")

# Cluster 级别多数投票 (排除 Ambiguous)
cluster_anno = {}
for cl in rna_merged.obs['leiden_0.5'].unique():
    mask = rna_merged.obs['leiden_0.5'] == cl
    votes = rna_merged.obs.loc[mask, 'predicted_celltype']
    non_ambig = votes[votes != 'Ambiguous']
    if len(non_ambig) > 0:
        top = non_ambig.value_counts()
        cluster_anno[cl] = top.index[0]
        # 报告投票比例
        ratio = top.iloc[0] / len(votes)
        if ratio < 0.5:
            print(f"  ⚠️ Cluster {cl}: {top.index[0]} 仅占 {ratio:.1%}, 第二名={top.index[1] if len(top) > 1 else 'N/A'}")
    else:
        cluster_anno[cl] = 'Unknown'
        print(f"  ⚠️ Cluster {cl}: 全部 Ambiguous → Unknown")

In [ ]:
rna_merged.obs['celltype'] = rna_merged.obs['leiden_0.5'].map(cluster_anno)

print("\n跨数据集细胞类型注释:")
for k, v in sorted(cluster_anno.items(), key=lambda x: int(x[0])):
    n = (rna_merged.obs['leiden_0.5'] == k).sum()
    print(f"  Cluster {k}: {v} ({n} cells)")

In [ ]:
# Marker dotplot
available = {}
for ct, genes in marker_gene_sets.items():
    present = [g for g in genes if g in rna_merged.var_names]
    if present:
        available[ct] = present

sc.pl.dotplot(rna_merged, var_names=available, groupby='celltype',
              standard_scale='var', dendrogram=True)
plt.savefig('./figures/1_snRNA_processing/merged_marker_dotplot.pdf', bbox_inches='tight')
plt.show()

In [ ]:

# cluster × marker gene dotplot
filtered = {
    ct: [g for g in genes if g in rna_merged.var_names]
    for ct, genes in marker_gene_sets.items()
}
filtered = {ct: genes for ct, genes in filtered.items() if genes}

sc.pl.dotplot(
    rna_merged, filtered, groupby='leiden_0.5',
    figsize=(36, 12), var_group_rotation=90,
    standard_scale='var', dendrogram=False, show=False,
)
plt.savefig('./figures/1_snRNA_processing/dotplot_markers_leiden.pdf', bbox_inches='tight')
plt.show()
print(f'✅ 已绘制 {len(filtered)} 组, 共 {sum(len(v) for v in filtered.values())} 个基因')


In [ ]:
# 细胞类型 UMAP
fig, axes = plt.subplots(1, 4, figsize=(21, 8))
sc.pl.umap(rna_merged, color='leiden_0.5', ax=axes[0], show=False,
           title='Clusters', legend_loc='on data', legend_fontsize=5)
sc.pl.umap(rna_merged, color='celltype', ax=axes[1], show=False,
           title='Cell Type (merged)', legend_fontsize=7)
sc.pl.umap(rna_merged, color='dataset', ax=axes[2], show=False,
           title='Dataset')
sc.pl.umap(rna_merged, color='tissue_unified', ax=axes[3], show=False,
           title='Tissue')

plt.tight_layout()
plt.savefig('./figures/1_snRNA_processing/merged_celltype_UMAP.pdf', bbox_inches='tight')
plt.show()

In [ ]:

from sklearn.neighbors import KernelDensity

def _repel_labels(positions, min_dist=1.5, iterations=50):
    """简单标签排斥: 让过近的标签互相推开"""
    pos = np.array(positions, dtype=float)
    for _ in range(iterations):
        moved = False
        for i in range(len(pos)):
            for j in range(i + 1, len(pos)):
                d = pos[j] - pos[i]
                dist = np.linalg.norm(d)
                if dist < min_dist and dist > 0:
                    shift = (min_dist - dist) / 2 * d / dist
                    pos[i] -= shift
                    pos[j] += shift
                    moved = True
        if not moved:
            break
    return pos

def enhance_umap(adata, color='cell_type', figsize=(8, 8), size=15,
                 kde_bandwidth=0.3, contour_level_frac=0.1,
                 label_fontsize=10, arrow_frac=0.10, palette=None,
                 label_min_dist=2.0, show_pct=True, show_arrows=True,
                 contour_color='gray', contour_style='--',
                 title=None):
    """融合: 生信大杂烩 + 生信星球 UMAP 美化"""
    fig, ax = plt.subplots(figsize=figsize)

    sc.pl.umap(adata, color=color, legend_loc=None, frameon=False,
               size=size, ax=ax, add_outline=True, palette=palette,
               outline_width=(0.1, 0.05), show=False)

    umap_coords = adata.obsm['X_umap']
    categories = adata.obs[color].cat.categories
    colors_list = adata.uns.get(f'{color}_colors', [f'C{i}' for i in range(len(categories))])

    # 细胞比例
    ct_counts = adata.obs[color].value_counts(normalize=True) * 100

    # 收集标签位置
    label_info = []
    for i, ct in enumerate(categories):
        mask = adata.obs[color] == ct
        coords = umap_coords[mask]
        if len(coords) < 30:
            continue
        mx, my = np.median(coords, axis=0)
        label_info.append((ct, mx, my, i, coords))

    # 标签排斥
    raw_positions = [(info[1], info[2]) for info in label_info]
    adjusted = _repel_labels(raw_positions, min_dist=label_min_dist)

    # KDE 轮廓 + 标签
    for idx, (ct, _, _, ci, coords) in enumerate(label_info):
        kde = KernelDensity(bandwidth=kde_bandwidth, metric='euclidean')
        kde.fit(coords)
        x_min_ct, x_max_ct = coords[:, 0].min() - 0.5, coords[:, 0].max() + 0.5
        y_min_ct, y_max_ct = coords[:, 1].min() - 0.5, coords[:, 1].max() + 0.5
        xx, yy = np.mgrid[x_min_ct:x_max_ct:100j, y_min_ct:y_max_ct:100j]
        xy_test = np.vstack([xx.ravel(), yy.ravel()]).T
        Z = np.exp(kde.score_samples(xy_test)).reshape(xx.shape)
        peak = Z.max()
        if peak > 0:
            ax.contour(xx, yy, Z, levels=[peak * contour_level_frac],
                       colors=contour_color, linestyles=contour_style,
                       linewidths=1.2, alpha=0.8)

        # 标签 (排斥后位置 + 可选百分比)
        lx, ly = adjusted[idx]
        pct = ct_counts.get(ct, 0)
        label_text = f"{ct} ({pct:.1f}%)" if show_pct else ct
        ax.text(lx, ly, label_text, fontsize=label_fontsize,
                ha='center', va='center', color='black',
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.7))

    # 科研风箭头坐标轴
    if show_arrows:
        x_all, y_all = umap_coords[:, 0], umap_coords[:, 1]
        x_lo, x_hi = x_all.min(), x_all.max()
        y_lo, y_hi = y_all.min(), y_all.max()
        x_range = x_hi - x_lo
        y_range = y_hi - y_lo
        ox = x_lo - x_range * 0.02
        oy = y_lo - y_range * 0.02
        dx = x_range * arrow_frac
        dy = y_range * arrow_frac
        ax.annotate('', xy=(ox + dx, oy), xytext=(ox, oy),
                    arrowprops=dict(arrowstyle='->', lw=1.5, color='black'))
        ax.annotate('', xy=(ox, oy + dy), xytext=(ox, oy),
                    arrowprops=dict(arrowstyle='->', lw=1.5, color='black'))
        ax.text(ox + dx / 2, oy - y_range * 0.03, 'UMAP1',
                fontsize=10, ha='center', va='top')
        ax.text(ox - x_range * 0.03, oy + dy / 2, 'UMAP2',
                fontsize=10, ha='right', va='center', rotation=90)

    # 标题
    if title is None:
        title = f"Overall (n = {adata.n_obs:,})"
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('')
    ax.set_ylabel('')
    plt.tight_layout()
    return fig, ax

print('enhance_umap 函数已定义 (融合版)')


In [ ]:

### CNV 分析 - 区分肿瘤细胞 vs 正常细胞
import infercnvpy as cnv

# 复制 adata，保留原始计数
adata_cnv = rna_merged.copy()

# 用 gencode v44 GTF 注释基因染色体位置（第一参数是 gtf_file）
cnv.io.genomic_position_from_gtf(
    './results/gencode.v44.basic.annotation.gtf.gz',
    adata_cnv,
    gtf_gene_id='gene_name',
)

print(f"var 列: {adata_cnv.var.columns.tolist()}")
print(f"有染色体信息的基因数: {adata_cnv.var['chromosome'].notna().sum()} / {adata_cnv.n_vars}")
print(adata_cnv.var[adata_cnv.var['chromosome'].notna()].head())


In [ ]:

# 每个 celltype 最多采样 500 个细胞（提高推断速度）
import numpy as np
np.random.seed(42)

n_per_type = 500
idx_list = []
for ct in adata_cnv.obs['celltype'].unique():
    idx = np.where(adata_cnv.obs['celltype'] == ct)[0]
    if len(idx) > n_per_type:
        idx = np.random.choice(idx, n_per_type, replace=False)
    idx_list.append(idx)

idx_all = np.concatenate(idx_list)
adata_sub = adata_cnv[idx_all].copy()
print(adata_sub.obs['celltype'].value_counts())
print(f"\n总细胞数: {adata_sub.n_obs}")


In [ ]:

# 运行 inferCNV（正常参考：Oligodendrocyte, Endothelial, Pericyte）
cnv.tl.infercnv(
    adata_sub,
    reference_key='celltype',
    reference_cat=['Oligodendrocyte', 'Endothelial', 'Pericyte'],
    window_size=250,
    n_jobs=4,
)
print("inferCNV 完成")
print(f"cnv 矩阵存储键: {list(adata_sub.obsm.keys())}")


In [ ]:

# 染色体热图 - 按 celltype 排序
cnv.pl.chromosome_heatmap(
    adata_sub,
    groupby='celltype',
    show=False,
)
plt.savefig('./figures/1_snRNA_processing/cnv_chromosome_heatmap.pdf', bbox_inches='tight')
plt.show()


In [ ]:

# 手动计算每个细胞的 CNV score（平均绝对 CNV 偏差）
adata_sub.obs['cnv_score'] = np.mean(np.abs(adata_sub.obsm['X_cnv']), axis=1)

# 在 CNV 空间运行降维（用 scanpy）
sc.pp.neighbors(adata_sub, use_rep='X_cnv', key_added='cnv')
sc.tl.umap(adata_sub, neighbors_key='cnv')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc.pl.umap(adata_sub, color='cnv_score', ax=axes[0], show=False,
           title='CNV Score (高=染色体异常多)', cmap='Reds')
sc.pl.umap(adata_sub, color='celltype', ax=axes[1], show=False,
           title='Cell Type')
plt.tight_layout()
plt.savefig('./figures/1_snRNA_processing/cnv_score_umap.pdf', bbox_inches='tight')
plt.show()

# 打印各 celltype 平均 CNV score
print("\n各 celltype 平均 CNV score（越高→染色体异常越多→越可能是肿瘤细胞）:")
print(adata_sub.obs.groupby('celltype')['cnv_score'].mean().sort_values(ascending=False).round(5))


In [ ]:

# OPC-like 内部 cluster CNV score 分布 + 关键 marker 基因表达
# OPC-like clusters: 1, 6, 11, 14

# 从原始数据中取 OPC-like 细胞（含 leiden 信息）
opc_mask = rna_merged.obs['celltype'] == 'OPC-like'
adata_opc = rna_merged[opc_mask].copy()

# 从 adata_sub 补回 cnv_score（subsample 中的 OPC-like）
opc_sub_mask = adata_sub.obs['celltype'] == 'OPC-like'
adata_opc_sub = adata_sub[opc_sub_mask]

print("OPC-like 各 cluster 的 CNV score (subsample):")
opc_cnv = adata_opc_sub.obs.groupby('leiden_0.5')['cnv_score'].agg(['mean', 'median', 'count'])
print(opc_cnv.sort_values('mean', ascending=False).round(5))

# 关键 marker 基因
normal_opc = ['PDGFRA', 'CSPG4', 'OLIG1', 'OLIG2', 'SOX10', 'VCAN']  # 正常 OPC
tumor_opc =  ['TOP2A', 'MKI67', 'CDK4', 'CCND2', 'EGFR', 'CDKN2A']   # 肿瘤/增殖标志
all_markers = [g for g in (normal_opc + tumor_opc) if g in rna_merged.var_names]

print(f"\n可用 marker 基因: {all_markers}")


In [ ]:

# OPC-like 各 cluster 的 marker 基因 dotplot（肿瘤 vs 正常区分）
# 正常 OPC: PDGFRA, OLIG1, OLIG2, SOX10, VCAN, CSPG4
# 肿瘤 OPC/增殖: TOP2A, MKI67, CDK4, CCND2, EGFR

# 只取 OPC-like 细胞 + Oligodendrocyte（对照）
compare_types = ['OPC-like', 'Oligodendrocyte']
adata_compare = rna_merged[rna_merged.obs['celltype'].isin(compare_types)].copy()

# 按 leiden_0.5 cluster 显示（只保留 OPC clusters + 部分 Oligo clusters 作参考）
opc_clusters = ['1', '6', '11', '14']
oligo_clusters = ['3', '8']   # Oligodendrocyte clusters
keep_clusters = opc_clusters + oligo_clusters
adata_compare = adata_compare[adata_compare.obs['leiden_0.5'].isin(keep_clusters)].copy()

markers_dict = {
    '正常OPC': ['OLIG1', 'OLIG2', 'SOX10', 'VCAN'],
    'PDGFRA': ['PDGFRA', 'CSPG4'],
    '增殖/肿瘤': ['TOP2A', 'MKI67', 'CDK4', 'CCND2'],
    'GBM特异': ['EGFR', 'CDKN2A'],
    'Oligo成熟': ['MOG', 'MBP', 'MOBP'],
}
markers_dict = {k: [g for g in v if g in rna_merged.var_names] for k, v in markers_dict.items()}

sc.pl.dotplot(
    adata_compare,
    var_names=markers_dict,
    groupby='leiden_0.5',
    standard_scale='var',
    title='OPC-like Clusters vs Oligodendrocyte: Tumor vs Normal Markers',
    save='_OPC_tumor_vs_normal.pdf',
)


In [ ]:

# 将 Cluster 11 (正常 OPC) 重新标注为 Oligodendrocyte
# 依据：CNV score 接近 Oligodendrocyte，所有 OPC/肿瘤 marker 均低表达

before = rna_merged.obs['celltype'].value_counts()
mask_cl11 = rna_merged.obs['leiden_0.5'] == '11'
rna_merged.obs['celltype'] = rna_merged.obs['celltype'].astype(str)
rna_merged.obs.loc[mask_cl11, 'celltype'] = 'Oligodendrocyte'
rna_merged.obs['celltype'] = rna_merged.obs['celltype'].astype('category')

after = rna_merged.obs['celltype'].value_counts()
print("修改前后 celltype 数量对比:")
print(f"OPC-like:       {before['OPC-like']} → {after['OPC-like']}")
print(f"Oligodendrocyte: {before['Oligodendrocyte']} → {after['Oligodendrocyte']}")
print(f"\nCluster 11 细胞数: {mask_cl11.sum()}")


In [ ]:

# 重新绘制 UMAP（更新后的 celltype 注释）
fig, ax = enhance_umap(
    rna_merged, color='celltype',
    figsize=(9, 8), size=12,
    kde_bandwidth=0.5, contour_level_frac=0.15,
    label_fontsize=9, label_min_dist=2.5,
    show_pct=True, show_arrows=True,
    title=f'snGBM Cell Types v2 (Cluster11→Oligo, n={rna_merged.n_obs:,})'
)
plt.savefig('./figures/1_snRNA_processing/enhanced_umap_celltype_v2.pdf', bbox_inches='tight')
plt.show()

print("\n最终 celltype 分布:")
print(rna_merged.obs['celltype'].value_counts())


In [ ]:

# 保存更新后的 h5ad（含修改的 celltype 注释）
import os
os.makedirs('./results/1_snRNA_processing', exist_ok=True)
rna_merged.write_h5ad('./results/1_snRNA_processing/adata1.h5ad')
print(f"已保存: adata1.h5ad")
print(f"细胞数: {rna_merged.n_obs}, 基因数: {rna_merged.n_vars}")
print(f"obs 列: {rna_merged.obs.columns.tolist()}")


### 4.5 细胞组成对比

In [ ]:
# 按数据集 × 组织类型 对比细胞组成
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# 按样本
ct_sample = pd.crosstab(rna_merged.obs['sample'], rna_merged.obs['celltype'],
                         normalize='index')
ct_sample.plot.bar(stacked=True, ax=axes[0], colormap='tab20')
axes[0].set_title('Cell Composition by Sample')
axes[0].legend(bbox_to_anchor=(1.05, 1), fontsize=6)
axes[0].set_ylabel('Proportion')

# 按数据集
ct_dataset = pd.crosstab(rna_merged.obs['dataset'], rna_merged.obs['celltype'],
                          normalize='index')
ct_dataset.plot.bar(stacked=True, ax=axes[1], colormap='tab20')
axes[1].set_title('Cell Composition by Dataset')
axes[1].legend(bbox_to_anchor=(1.05, 1), fontsize=7)

# 按组织（统一标签）- 转 str 避免 Categorical + str 报错
ct_tissue = pd.crosstab(
    rna_merged.obs['dataset'].astype(str) + '_' + rna_merged.obs['tissue_unified'].astype(str),
    rna_merged.obs['celltype'], normalize='index')
ct_tissue.plot.bar(stacked=True, ax=axes[2], colormap='tab20')
axes[2].set_title('Cell Composition: Dataset × Tissue')
axes[2].legend(bbox_to_anchor=(1.05, 1), fontsize=6)

plt.tight_layout()
plt.savefig('./figures/1_snRNA_processing/merged_cell_composition.pdf', bbox_inches='tight', dpi=150)
plt.show()

### 4.6 Tumor vs Peritumoral 差异分析

In [ ]:
# 在合并数据上做 Tumor vs Peritumoral 差异分析
sc.tl.rank_genes_groups(rna_merged, groupby='tissue_unified', method='wilcoxon',
                        key_added='tissue_deg_merged')

sc.pl.rank_genes_groups(rna_merged, n_genes=15, key='tissue_deg_merged', sharey=False)
plt.savefig('./figures/1_snRNA_processing/merged_DEG_tissue.pdf', bbox_inches='tight')
plt.show()

deg_merged = sc.get.rank_genes_groups_df(rna_merged, group='Tumor',
                                          key='tissue_deg_merged')
deg_sig_m = deg_merged[deg_merged['pvals_adj'] < 0.05]
print(f"跨数据集 DEG (Tumor vs Peritumoral): {len(deg_sig_m)}")
print(f"上调: {(deg_sig_m['logfoldchanges'] > 0.5).sum()}")
print(f"下调: {(deg_sig_m['logfoldchanges'] < -0.5).sum()}")

In [ ]:
# 细胞类型内的差异分析
# 用整数索引 + use_raw=False 避免 .raw/scipy 索引兼容问题
for ct in rna_merged.obs['celltype'].value_counts().index[:5]:
    mask = rna_merged.obs['celltype'] == ct
    idx = np.where(mask.values)[0]
    sub = ad.AnnData(
        X=rna_merged.X[idx].copy(),
        obs=rna_merged.obs.iloc[idx].copy(),
        var=rna_merged.var.copy()
    )
    if sub.obs['tissue_unified'].nunique() < 2:
        continue
    if (sub.obs['tissue_unified'] == 'Tumor').sum() < 10:
        continue
    sc.tl.rank_genes_groups(sub, groupby='tissue_unified', method='wilcoxon',
                            key_added='tissue_deg', use_raw=False)
    result = sc.get.rank_genes_groups_df(sub, group='Tumor', key='tissue_deg')
    sig = result[result['pvals_adj'] < 0.05]
    up = (sig['logfoldchanges'] > 0.5).sum()
    down = (sig['logfoldchanges'] < -0.5).sum()
    print(f"  {ct}: {len(sig)} DEG (↑{up} / ↓{down})")

In [ ]:
# 保存 snRNA 整合结果
rna_merged.write('./results/1_snRNA_processing/adata1.h5ad')
print(f"✓ snRNA 整合保存: {rna_merged.n_obs} cells × {rna_merged.n_vars} genes")